# 데이터보관최초일자 빠른 재계산 (Jupyter)

3억 row 테이블에서 `SELECT MIN(날짜)` 풀스캔이 느린 문제를 해결합니다.
기존 완성 Excel 의 `데이터보관최초일자` 컬럼만 **빠른 방식으로 다시 계산**해 새 파일에 기록합니다.

**계산 우선순위 (테이블당)**
1. `num_rows = 0` → `-`
2. 날짜컬럼 없음 → 테이블 생성일(`user_objects.created`)
3. **DATE 날짜컬럼** → `user_tab_col_statistics.low_value` 디코딩 (스캔 0, 즉시) ← 핵심
4. **VARCHAR 날짜컬럼** → 통계 low_value 문자열 디코딩 → 날짜 정규화
5. 통계 없음 / TIMESTAMP / 디코딩 실패 → `MIN()` 폴백 *(MIN_FALLBACK=False 면 생성일로 대체)*

> 파티션 경계값(방법2)은 테이블별 파티션 구조에 의존도가 커서 노트북엔 넣지 않았습니다.
> 날짜 RANGE 파티션 테이블은 `fast_first_date_test.sql` 의 방법2 로 개별 확인하세요.
> **원본은 수정하지 않고** `원본명_최초일자갱신.xlsx` 새 파일을 만듭니다.

| 셀 | 내용 | 비고 |
|---|---|---|
| 1 | 임포트 | |
| 2 | **설정 ★ 수정 필요** | 파일경로·DB·옵션 |
| 3 | 유틸 + 날짜컬럼 탐지 + 빠른 최초일자 | |
| 4 | 원본 Excel 읽기 + 버전 자동판별 | |
| 5 | Oracle 접속 + 테이블 메타 로드 | num_rows·생성일 |
| 6 | 최초일자 일괄 계산 + 체크포인트 저장 | ⏳ |
| 7 | 체크포인트 로드 (선택) | 재실행 시 셀 6 건너뛰기 |
| 8 | 새 파일 생성·기록 | 원본 보존 |
| 9 | 검증 리포트 | 방식별 집계 |

In [ ]:
# 셀 1: 임포트
import os, re, shutil, pickle
from datetime import datetime

import oracledb
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string
from dateutil import parser as dateutil_parser

print('imports OK')

In [ ]:
# 셀 2: 설정 ★ 아래 값을 실제 환경에 맞게 수정하세요
ORACLE_CLIENT_LIB = r'C:\oracle\instantclient_21_9'  # ★

PROD_HOST     = 'PROD_HOST'      # ★
PROD_PORT     = 1521
PROD_SERVICE  = 'PROD_SERVICE'   # ★
PROD_USER     = 'PROD_USER'      # ★
PROD_PASSWORD = 'PROD_PASSWORD'  # ★

ARCH_HOST     = 'ARCHIVE_HOST'     # ★ v2 만
ARCH_PORT     = 1521
ARCH_SERVICE  = 'ARCHIVE_SERVICE'  # ★ v2 만
ARCH_USER     = 'ARCHIVE_USER'     # ★ v2 만
ARCH_PASSWORD = 'ARCHIVE_PASSWORD' # ★ v2 만

INFA_HOST     = 'INFA_HOST'      # ★ v2 만
INFA_PORT     = 1521
INFA_SERVICE  = 'INFA_SERVICE'   # ★ v2 만
INFA_USER     = 'INFA_USER'      # ★ v2 만
INFA_PASSWORD = 'INFA_PASSWORD'  # ★ v2 만

SOURCE_EXCEL = r'C:\path\to\완성본.xlsx'  # ★ (수정 안 됨)
START_CELL   = 'B4'
VERSION      = 'auto'   # 'auto' | 'v1' | 'v2'

# 통계 low_value 가 없거나 디코딩 실패할 때 MIN() 풀스캔으로 폴백할지
#  True  = 정확하지만 느린 테이블이 섞일 수 있음
#  False = 폴백 안 함 → 생성일로 대체하고 리포트에 표시(완전 무스캔)
MIN_FALLBACK = True

# 통계 수집 경과일이 이 값을 넘으면 리포트에 'STALE' 경고 (계산은 그대로 사용)
STATS_WARN_AGE_DAYS = 30

CHECKPOINT_FILE = 'fast_first_date_checkpoint.pkl'
OUTPUT_EXCEL    = re.sub(r'\.xlsx$', '_최초일자갱신.xlsx', SOURCE_EXCEL)
print(f'원본 : {SOURCE_EXCEL}')
print(f'출력 : {OUTPUT_EXCEL}')
print(f'MIN 폴백: {MIN_FALLBACK}')

In [ ]:
# 셀 3: 유틸 + 날짜컬럼 탐지 + 빠른 최초일자
CREATE_PATTERNS = re.compile(r'CRE|REG|INS|FIRST|OPEN|START|BEGIN|OCCUR|INIT|ENTR',
                             re.IGNORECASE)
VARCHAR_DATE_PATTERNS = [
    (8,  r'^\d{8}$'),  (12, r'^\d{12}$'), (14, r'^\d{14}$'),
    (16, r'^\d{16}$'), (17, r'^\d{17,}$'),
    (10, r'^\d{4}-\d{2}-\d{2}$'),
    (19, r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'),
]

def parse_cell(ref):
    m = re.match(r'^([A-Za-z]+)(\d+)$', ref.strip())
    if not m:
        raise ValueError(f'잘못된 셀 참조: {ref}')
    return column_index_from_string(m.group(1)), int(m.group(2))

def norm_date_str(v):
    """DATE/문자 값 → 'YYYY-MM-DD'. 실패 시 앞 10자."""
    if v is None:
        return None
    if isinstance(v, datetime):
        return v.strftime('%Y-%m-%d')
    s = str(v).strip()
    if not s:
        return None
    if re.match(r'^\d{8}', s):
        try:
            return datetime.strptime(s[:8], '%Y%m%d').strftime('%Y-%m-%d')
        except Exception:
            pass
    try:
        return dateutil_parser.parse(s, yearfirst=True, ignoretz=True).strftime('%Y-%m-%d')
    except Exception:
        return s[:10]

def _try_vc_date(val):
    s = str(val).strip()
    for min_len, pat in VARCHAR_DATE_PATTERNS:
        if len(s) >= min_len and re.match(pat, s):
            return True
    try:
        dateutil_parser.parse(s, yearfirst=True, ignoretz=True)
        return True
    except Exception:
        return False

def _is_vc_date_col(conn, tbl, col):
    cur = conn.cursor()
    try:
        cur.execute(f'SELECT {col} FROM {tbl} WHERE ROWNUM <= 100')
        vals = [r[0] for r in cur.fetchall() if r[0] is not None]
        if not vals:
            return False
        return sum(1 for v in vals if _try_vc_date(str(v))) / len(vals) >= 0.8
    except Exception:
        return False
    finally:
        cur.close()

def detect_date_column(conn, tbl):
    """(컬럼명, 데이터타입) 반환. 없으면 (None, None). survey 와 동일 우선순위."""
    cur = conn.cursor()
    cur.execute('''SELECT column_name, data_type FROM user_tab_columns
                   WHERE table_name = :1 ORDER BY column_id''', [tbl])
    cols = cur.fetchall()
    cur.close()
    date_pat, date_plain, vc = [], [], []
    for cn, dt in cols:
        if dt == 'DATE' or dt.startswith('TIMESTAMP'):
            (date_pat if CREATE_PATTERNS.search(cn) else date_plain).append((cn, dt))
        elif 'CHAR' in dt:
            vc.append((cn, dt))
    if date_pat:
        return date_pat[0]
    if date_plain:
        return date_plain[0]
    for cn, dt in vc:
        if _is_vc_date_col(conn, tbl, cn):
            return (cn, dt)
    return (None, None)

# ── 통계 low_value 디코딩 (DATE / VARCHAR) — 테이블 스캔 없음 ──
PLSQL_DATE = '''
DECLARE
  r   user_tab_col_statistics.low_value%TYPE;
  d   DATE;
  la  DATE;
BEGIN
  SELECT low_value, last_analyzed INTO r, la
  FROM   user_tab_col_statistics
  WHERE  table_name = :1 AND column_name = :2;
  IF r IS NULL THEN :3 := NULL; :4 := NULL; RETURN; END IF;
  DBMS_STATS.CONVERT_RAW_VALUE(r, d);
  :3 := TO_CHAR(d, 'YYYY-MM-DD');
  :4 := TO_CHAR(la, 'YYYY-MM-DD');
EXCEPTION
  WHEN NO_DATA_FOUND THEN :3 := NULL; :4 := NULL;
  WHEN OTHERS         THEN :3 := NULL; :4 := NULL;
END;
'''

PLSQL_VC = '''
DECLARE
  r   user_tab_col_statistics.low_value%TYPE;
  v   VARCHAR2(200);
  la  DATE;
BEGIN
  SELECT low_value, last_analyzed INTO r, la
  FROM   user_tab_col_statistics
  WHERE  table_name = :1 AND column_name = :2;
  IF r IS NULL THEN :3 := NULL; :4 := NULL; RETURN; END IF;
  DBMS_STATS.CONVERT_RAW_VALUE(r, v);
  :3 := v;
  :4 := TO_CHAR(la, 'YYYY-MM-DD');
EXCEPTION
  WHEN NO_DATA_FOUND THEN :3 := NULL; :4 := NULL;
  WHEN OTHERS         THEN :3 := NULL; :4 := NULL;
END;
'''

def stats_low_value(conn, tbl, col, is_varchar):
    """(정규화된 'YYYY-MM-DD' or None, last_analyzed str or None)"""
    cur = conn.cursor()
    try:
        o_val = cur.var(oracledb.DB_TYPE_VARCHAR, 200)
        o_la  = cur.var(oracledb.DB_TYPE_VARCHAR, 20)
        cur.execute(PLSQL_VC if is_varchar else PLSQL_DATE,
                    [tbl, col, o_val, o_la])
        raw, la = o_val.getvalue(), o_la.getvalue()
        if raw is None:
            return None, None
        return norm_date_str(raw), la
    except Exception:
        return None, None
    finally:
        cur.close()

def min_scan(conn, tbl, col):
    """폴백: 직접 MIN() (인덱스 없으면 느릴 수 있음)"""
    cur = conn.cursor()
    try:
        cur.execute(f'SELECT MIN({col}) FROM {tbl}')
        row = cur.fetchone()
        return norm_date_str(row[0]) if row and row[0] is not None else None
    except Exception:
        return None
    finally:
        cur.close()

def fast_first_date(conn, tbl, num_rows, created):
    """returns (값, 방식, last_analyzed)  방식 ∈ EMPTY/CREATED/STATS/STATS-VC/MIN/MISS"""
    if num_rows == 0:
        return '-', 'EMPTY', None
    col, dt = detect_date_column(conn, tbl)
    if col is None:
        return (norm_date_str(created) or '-'), 'CREATED', None

    is_vc = (dt is not None) and ('CHAR' in dt)
    is_ts = (dt is not None) and dt.startswith('TIMESTAMP')

    # TIMESTAMP 는 CONVERT_RAW_VALUE 미지원 → 통계 건너뜀
    if not is_ts:
        val, la = stats_low_value(conn, tbl, col, is_vc)
        if val:
            return val, ('STATS-VC' if is_vc else 'STATS'), la

    if MIN_FALLBACK:
        v = min_scan(conn, tbl, col)
        if v:
            return v, 'MIN', None
    return (norm_date_str(created) or '-'), 'MISS', None

print('utilities OK')

In [ ]:
# 셀 4: 원본 Excel 읽기 + 버전 자동판별 (읽기 전용)
if not os.path.isfile(SOURCE_EXCEL):
    raise FileNotFoundError(f'파일 없음: {SOURCE_EXCEL}')

start_col, start_row = parse_cell(START_CELL)
_wb = load_workbook(SOURCE_EXCEL, read_only=True, data_only=True)
_ws = _wb.active
DATA_SHEET = _ws.title

def detect_version():
    if VERSION in ('v1', 'v2'):
        return VERSION
    for rr in range(max(1, start_row - 5), start_row):
        for cc in range(start_col, start_col + 10):
            v = _ws.cell(row=rr, column=cc).value
            if v is not None and str(v).strip() == '소재':
                return 'v2'
    return 'v1'

ver = detect_version()
if ver == 'v2':
    OFF_NO, OFF_LOC, OFF_TBL, OFF_DATE = 0, 2, 3, 6
else:
    OFF_NO, OFF_LOC, OFF_TBL, OFF_DATE = 0, None, 2, 5

records = []
rr = start_row
while True:
    no_val = _ws.cell(row=rr, column=start_col + OFF_NO).value
    if no_val is None or str(no_val).strip() == '':
        break
    tbl = _ws.cell(row=rr, column=start_col + OFF_TBL).value
    loc = (_ws.cell(row=rr, column=start_col + OFF_LOC).value
           if OFF_LOC is not None else None)
    records.append({
        'excel_row':  rr,
        'table_name': str(tbl).strip() if tbl is not None else '',
        'location':   str(loc).strip() if loc is not None else None,
    })
    rr += 1
_wb.close()
print(f'버전: {ver}  데이터시트: {DATA_SHEET}  레코드: {len(records)}개')
print('예시:', records[:3])

In [ ]:
# 셀 5: Oracle 접속 + 테이블 메타(num_rows·생성일) 로드
oracledb.init_oracle_client(lib_dir=ORACLE_CLIENT_LIB)

META_SQL = '''
    SELECT t.table_name, t.num_rows, o.created
    FROM   user_tables t
    LEFT JOIN user_objects o
           ON o.object_name = t.table_name AND o.object_type = 'TABLE'
'''

def connect(user, pw, dsn, label):
    try:
        c = oracledb.connect(user=user, password=pw, dsn=dsn)
        print(f'{label} 접속 성공')
        return c
    except Exception as e:
        print(f'{label} 접속 실패 (건너뜀): {e}')
        return None

def load_meta(conn):
    if conn is None:
        return {}
    cur = conn.cursor()
    cur.execute(META_SQL)
    m = {}
    for tn, nr, cr in cur.fetchall():
        m[tn] = {'num_rows': int(nr) if nr is not None else None, 'created': cr}
    cur.close()
    return m

conn_prod = connect(PROD_USER, PROD_PASSWORD,
                    f'{PROD_HOST}:{PROD_PORT}/{PROD_SERVICE}', 'PROD')
conn_arch = conn_infa = None
if ver == 'v2':
    conn_arch = connect(ARCH_USER, ARCH_PASSWORD,
                        f'{ARCH_HOST}:{ARCH_PORT}/{ARCH_SERVICE}', 'ARCHIVE')
    conn_infa = connect(INFA_USER, INFA_PASSWORD,
                        f'{INFA_HOST}:{INFA_PORT}/{INFA_SERVICE}', 'INFA')

meta_prod = load_meta(conn_prod)
meta_arch = load_meta(conn_arch)
meta_infa = load_meta(conn_infa)
print(f'메타 로드: PROD {len(meta_prod)} / ARCH {len(meta_arch)} / INFA {len(meta_infa)}')

In [ ]:
# 셀 6: 최초일자 일괄 계산 (⏳ — STATS 는 즉시, MIN 폴백 테이블만 느림)
def pick(rec):
    """소재에 따라 (conn, meta) 선택. 없으면 PROD 폴백."""
    loc = rec['location']
    if loc == 'INFA' and conn_infa:
        return conn_infa, meta_infa
    if loc == 'Archive Only' and conn_arch:
        return conn_arch, meta_arch
    return conn_prod, meta_prod

results = {}   # excel_row -> (value, method, last_analyzed)
total = len(records)
for i, rec in enumerate(records, 1):
    conn, meta = pick(rec)
    tbl = rec['table_name']
    if conn is None or tbl not in meta:
        results[rec['excel_row']] = (None, 'NO-TABLE', None)
    else:
        m = meta[tbl]
        nr = m['num_rows'] if m['num_rows'] is not None else -1
        results[rec['excel_row']] = fast_first_date(conn, tbl, nr, m['created'])
    if i % 100 == 0 or i == total:
        v, mth, _ = results[rec['excel_row']]
        print(f'  {i}/{total}  {tbl} → {v} [{mth}]')

with open(CHECKPOINT_FILE, 'wb') as f:
    pickle.dump(results, f)
print(f'체크포인트 저장: {CHECKPOINT_FILE}  ({len(results)}건)')

In [ ]:
# 셀 7: 체크포인트 로드 (선택 — 재실행 시 셀 5~6 건너뛰고 이 셀만)
with open(CHECKPOINT_FILE, 'rb') as f:
    results = pickle.load(f)
print(f'로드 완료: {len(results)}건')

In [ ]:
# 셀 8: 새 파일 생성 → 데이터보관최초일자 컬럼만 갱신 → 저장 (원본 보존)
shutil.copy2(SOURCE_EXCEL, OUTPUT_EXCEL)
print(f'복사 완료: {OUTPUT_EXCEL}')

wb = load_workbook(OUTPUT_EXCEL)
ws = wb[DATA_SHEET]

updated = skipped = 0
for rec in records:
    xrow = rec['excel_row']
    val, method, _ = results.get(xrow, (None, 'NO-RESULT', None))
    if val is None:
        skipped += 1
        continue
    ws.cell(row=xrow, column=start_col + OFF_DATE).value = val
    updated += 1

wb.save(OUTPUT_EXCEL)
print(f'저장 완료: {OUTPUT_EXCEL}')
print(f'갱신 {updated}건 / 미갱신(원본유지) {skipped}건')

In [ ]:
# 셀 9: 검증 리포트 — 방식별 집계 + STALE/MIN 점검
from collections import Counter

by_method = Counter()
stale, min_used, no_table = [], [], []
today = datetime.now()

for rec in records:
    xrow = rec['excel_row']
    val, method, la = results.get(xrow, (None, 'NO-RESULT', None))
    by_method[method] += 1
    if method == 'MIN':
        min_used.append(rec['table_name'])
    if method == 'NO-TABLE':
        no_table.append(rec['table_name'])
    if la:
        try:
            age = (today - datetime.strptime(la, '%Y-%m-%d')).days
            if age > STATS_WARN_AGE_DAYS:
                stale.append((rec['table_name'], la, age))
        except Exception:
            pass

print('=== 방식별 집계 ===')
for k, v in by_method.most_common():
    print(f'  {k:10s}: {v}')
print(f'\n원본(보존): {SOURCE_EXCEL}')
print(f'출력      : {OUTPUT_EXCEL}')

if min_used:
    print(f'\n[MIN 폴백 {len(min_used)}건] 통계 없어 풀스캔함 — 통계수집 권장:')
    for t in min_used[:30]:
        print(f'  {t}')
    if len(min_used) > 30:
        print(f'  ... 외 {len(min_used) - 30}건')

if stale:
    print(f'\n[STALE {len(stale)}건] 통계 {STATS_WARN_AGE_DAYS}일 초과 — 값이 과거에 치우칠 수 있음:')
    for t, la, age in stale[:30]:
        print(f'  {t}  (last_analyzed={la}, {age}일 전)')
    if len(stale) > 30:
        print(f'  ... 외 {len(stale) - 30}건')

if no_table:
    print(f'\n[테이블없음 {len(no_table)}건] 해당 DB 에서 못 찾음 — 원본값 유지:')
    for t in no_table[:30]:
        print(f'  {t}')

if not min_used and not stale and not no_table:
    print('\n경고 없음 — 전 건 통계/생성일로 즉시 처리')